In [11]:
import os
import json
import time
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("Groq API Key Loaded ✅" if os.getenv("GROQ_API_KEY") else "❌ Key not found")

Groq API Key Loaded ✅


In [12]:
sample_answers = [
    "The Eiffel Tower was built in 1889 and stands 330 metres tall. It was designed by Gustave Eiffel and is located in Paris, France.",
    "Photosynthesis is a process used by plants to convert sunlight into food. It occurs in the chloroplasts and requires carbon dioxide and water. Oxygen is released as a byproduct.",
    "Python was created by Guido van Rossum and first released in 1991. It is known for its simple syntax and is widely used in data science and AI."
]
print(f"Total test answers: {len(sample_answers)}")

Total test answers: 3


In [13]:
def extract_claims(answer: str) -> list[dict]:
    """
    Takes an answer string.
    Returns a list of atomic factual claims as dicts.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a claim decomposition engine for AI evaluation.
Break down any given text into individual atomic factual claims.

Rules:
- Each claim must contain ONE fact only
- Each claim must be self-contained and understandable alone
- Each claim must be a declarative statement
- Do NOT include opinions or vague statements
- Do NOT merge two facts into one claim

Return ONLY a valid JSON array. No explanation. No markdown.
Format: [{"claim_id": 1, "text": "..."}, {"claim_id": 2, "text": "..."}]"""
            },
            {
                "role": "user",
                "content": f"Decompose this answer into atomic claims:\n\n{answer}"
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    
    parsed = json.loads(raw)

    # Handle both {"claims": [...]} and direct [...] responses
    if isinstance(parsed, dict):
        claims = list(parsed.values())[0]
    else:
        claims = parsed

    return claims

In [14]:
for i, answer in enumerate(sample_answers):
    print(f"\n{'='*60}")
    print(f"ANSWER {i+1}:")
    print(f"{answer}")
    print(f"\nEXTRACTED CLAIMS:")

    claims = extract_claims(answer)
    for claim in claims:
        print(f"  [{claim['claim_id']}] {claim['text']}")


ANSWER 1:
The Eiffel Tower was built in 1889 and stands 330 metres tall. It was designed by Gustave Eiffel and is located in Paris, France.

EXTRACTED CLAIMS:
  [1] The Eiffel Tower was built in 1889
  [2] The Eiffel Tower stands 330 metres tall
  [3] The Eiffel Tower was designed by Gustave Eiffel
  [4] The Eiffel Tower is located in Paris, France

ANSWER 2:
Photosynthesis is a process used by plants to convert sunlight into food. It occurs in the chloroplasts and requires carbon dioxide and water. Oxygen is released as a byproduct.

EXTRACTED CLAIMS:
  [1] Photosynthesis is a process used by plants to convert sunlight into food.
  [2] Photosynthesis occurs in the chloroplasts.
  [3] Photosynthesis requires carbon dioxide and water.
  [4] Oxygen is released as a byproduct of photosynthesis.

ANSWER 3:
Python was created by Guido van Rossum and first released in 1991. It is known for its simple syntax and is widely used in data science and AI.

EXTRACTED CLAIMS:
  [1] Python was creat

In [15]:
print("✅ Day 1 Complete!")
print("Next: Copy extract_claims() into verifaith/claim_extractor.py")
print("Then: Build the Span Retriever on Day 2")

✅ Day 1 Complete!
Next: Copy extract_claims() into verifaith/claim_extractor.py
Then: Build the Span Retriever on Day 2
